## ERA5 weather features — already included, nothing to attach

The repo carries `data/raw/era5/reservoir_era5_daily.csv` — the per-reservoir
daily extraction of ERA5 **surface runoff (sro), evaporation (e), soil moisture (swvl1)**
for all 10 dams (2010–2024, no gaps). `main.py` loads it automatically.

To measure what weather adds (the ablation): run the seed loop as-is (weather ON),
then optionally re-run with the CSV temporarily renamed — the difference in
held-out NSE is the 'weather features add X' result.

Note: the ERA5 download contains no `tp` (total precipitation) band, so the
meteorological slot uses surface runoff instead of rainfall. Adding `tp` later
means a fresh CDS download + re-running `scripts/extract_era5_points.py`.

In [ ]:
# 1. Get the code + data into the writable working dir
#    (public repo -> plain git clone works; attached inputs are used if present)
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rajhodedara/gnn-reservoirnet.git"
dst = Path("/kaggle/working/gnn-reservoirnet")

src = next((c for c in [
    "/kaggle/input/gnn-reservoirnet",
    "/kaggle/input/gnn-reservoirnet-dataset",
] if Path(c).exists()), None)

if dst.exists():
    shutil.rmtree(dst)

if src:
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints"))
    print("Copied attached input:", src)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dst)],
                       capture_output=True, text=True)
    assert r.returncode == 0, f"git clone failed: {r.stderr}"
    print("Cloned public repo:", REPO_URL)

os.chdir(dst)
print("Working in:", os.getcwd())

In [ ]:
# 2. Install only what Kaggle lacks (torch is preinstalled)
#    captum is required: main.py imports the IG explainer at module level
!pip install -q torch-geometric properscoring captum

In [ ]:
# 3. Sanity checks: dataset complete + IOD present
import glob
import pandas as pd

files = sorted(glob.glob("data/raw/wris_v2/*.csv"))
assert len(files) == 10, f"Expected 10 reservoir CSVs, found {len(files)}: {files}"
cli = pd.read_csv("data/raw/enso/combined_climate_indices.csv")
assert "iod" in cli.columns, "IOD missing — the climate CSV is stale; re-merge with scripts/merge_iod.py"
for f in files:
    n = sum(1 for _ in open(f, encoding="utf-8")) - 1
    assert n == 5479, f"{f}: {n} rows != 5479"
print("Data OK:", len(files), "reservoirs x 5479 days; climate columns:", list(cli.columns))

In [ ]:
# 4. Smoke-check the test suite on the Kaggle environment (optional but recommended)
!python -m pytest tests/ -q --no-header

In [ ]:
# 5. TRAIN — 3 seeds (mean ± std turns 'looks better' into 'statistically better')
#    Each run early-stops on its own (~2 min each on T4). Trains 2010-2022,
#    validates on 2023, then evaluates BOTH val and held-out test (2024).
import shutil

SEEDS = [42, 7, 123]
for seed in SEEDS:
    print(f"\n================ SEED {seed} ================")
    !python main.py --config configs/default_config.yaml --seed {seed}
    seed_dir = Path(f"runs/seed{seed}")
    seed_dir.mkdir(parents=True, exist_ok=True)
    for f in [
        "best_model_finetune.pt",
        "evaluation_metrics_per_reservoir.csv", "evaluation_metrics_per_basin.csv",
        "evaluation_metrics_enso.csv",
        "evaluation_metrics_per_reservoir_test.csv", "evaluation_metrics_per_basin_test.csv",
        "evaluation_metrics_enso_test.csv",
        "evaluation_metrics_by_week.csv", "evaluation_metrics_by_week_test.csv",
        "explainability_report.json",
    ]:
        src = Path("runs") / f
        if src.exists():
            shutil.move(str(src), str(seed_dir / f))
print("\nAll seeds trained and archived under runs/seed*/")

## Ablation & diagnostics (Tier-2)

ERA5 weather features (runoff/evap/soil-moisture at each dam, 2010–2024)
are **committed in the repo** and auto-loaded by `main.py` — nothing to attach.

The cells below run the two Tier-2 analyses:
- **blending**: per-(week, reservoir) optimal blend of GNN predictions with
  seasonal climatology — weights fitted on val (2023), applied to test (2024);
- **node diagnostics**: lag-1 autocorrelation, climatology R², zero% per
  reservoir — explains each node's scoreboard position.

To measure what weather adds, re-run the seed loop with
`data/raw/era5/reservoir_era5_daily.csv` temporarily renamed (weather OFF)
and compare the two `evaluation_metrics_per_reservoir_test.csv` files.

In [ ]:
# 6. Aggregate the 3 seeds -> mean ± std per reservoir (held-out 2024)
!python scripts/aggregate_seeds.py

In [ ]:
# 7. Recompute the baselines in the same environment for a fair comparison
!python scripts/run_baselines.py --config configs/default_config.yaml

In [ ]:
# 7. Long-lead blending: GNN + climatology, weights fitted on val (2023)
!python scripts/blend_eval.py --seed-dir seed42

In [ ]:
# 9. Node predictability diagnostics (lag-1 autocorr, climatology R^2, zero%)
!python scripts/diagnose_nodes.py

In [ ]:
# 10. Artifact inventory — /kaggle/working is saved as the notebook output
from pathlib import Path

for p in sorted(Path("runs").rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size:>10,}  {p}")
print("\nDownload runs/ + outputs/ from the notebook Output tab, then hand them back for verification.")

In [ ]:
# 11. Package artifacts as .zip for one-click download
import shutil
from pathlib import Path

for folder in ["runs", "outputs"]:
    src = Path(folder)
    if src.exists() and any(src.iterdir()):
        zpath = shutil.make_archive(f"artifacts_{folder}", "zip", root_dir=".", base_dir=folder)
        size_mb = Path(zpath).stat().st_size / 1e6
        print(f"created {zpath} ({size_mb:.1f} MB)")
print("\nDownload artifacts_runs.zip / artifacts_outputs.zip from the Output tab.")